# Diffusion Distance Based Clustering

Execution notebook using the reusable functions in `ddbc_functions.py`.

# Import

In [ ]:
%load_ext autoreload
%autoreload 2

import gc
import os
import pickle
import matplotlib.pyplot as plt
import numpy as np

from ddbc_functions import (
    DDBCConfig,
    big_objects,
    build_column_aggregation_matrix,
    build_interlayer_coupling_matrices,
    build_kNN,
    build_layer_adjacency_matrices,
    build_multilayer_transition_matrix,
    build_row_aggregation_matrix,
    build_distinct_gene_mapping,
    compute_and_save_diffusion_distance,
    compute_and_save_average_transition_matrix,
    compute_stationary_layer_weights,
    load_bootstrap_scores,
    load_diffusion_distance,
    load_matrices,
    plot_stationary_layer_weights,
    run_bootstrap_robustness,
    run_clustering,
    select_communities,
    stationary_distribution,
)

# Configuration

In [ ]:
DISEASE = input("Disease: ")

config = DDBCConfig(
    disease=DISEASE,
    average_t=[2, 4, 6, 8],
    interlayer_transition_prob=0.35,
    num_neighbors=400,
    resolution=1.3,
    size_cap=100,
    score_cap=0,
    n_boots=1000,
    check_every=50,
    tol=0.01,
    patience=2,
    sampling_pct=0.8,
    leiden_seed=42,
)

# Building Multilayer Transition Matrix

In [ ]:
os.makedirs(config.output_directory, exist_ok=True)

matrices = load_matrices(config)
DGIDB_adjacency_matrix, MSIGDB_adjacency_matrix = (
    build_layer_adjacency_matrices(matrices)
)

## Coupling matrices and distinct gene indices

In [ ]:
coupling = build_interlayer_coupling_matrices(
    DGIDB_adjacency_matrix,
    MSIGDB_adjacency_matrix,
    config,
)

gene_to_index_distinct = build_distinct_gene_mapping(coupling, config)

## Construct the multilayer transition matrix

In [ ]:
P = build_multilayer_transition_matrix(
    DGIDB_adjacency_matrix,
    MSIGDB_adjacency_matrix,
    coupling,
)
num_genes = P.shape[0]

del DGIDB_adjacency_matrix, MSIGDB_adjacency_matrix
for _ in range(3):
    gc.collect()

## Stationary distribution

In [ ]:
pi = stationary_distribution(
    P,
    tol=config.stationary_tol,
    maxit=config.stationary_maxit,
    seed=config.stationary_seed,
)
pi

# Aggregation

## Column aggregation

In [ ]:
A_c = build_column_aggregation_matrix(
    P,
    coupling,
    gene_to_index_distinct,
)

## Preparing weights

In [ ]:
wD_list, wM_list = compute_stationary_layer_weights(pi, coupling)
plot_stationary_layer_weights(wD_list, wM_list, config)

## Row aggregation

In [ ]:
A_r, wD_list2, wM_list2 = build_row_aggregation_matrix(
    P,
    pi,
    coupling,
    gene_to_index_distinct,
)

assert wD_list == wD_list2 and wM_list == wM_list2

# Matrix-Free Method

In [ ]:
P_t_agg_avg = compute_and_save_average_transition_matrix(
    P,
    config,
    A_r,
    A_c,
)

big_objects()

In [ ]:
D_avg = compute_and_save_diffusion_distance(P_t_agg_avg, config)

In [ ]:
del P_t_agg_avg
gc.collect()

# Clustering Methods

## Constructing kNN

In [ ]:
if "D_avg" not in locals():
    D_avg = load_diffusion_distance(config)

kNN_adjacency_matrix, kNN_graph = build_kNN(
    D_avg,
    k=config.num_neighbors,
    sym_method="average",
)
kNN_adjacency_matrix.data.mean()

In [ ]:
with open(f"{config.output_directory}/result_graph.pkl", "wb") as file:
    pickle.dump(kNN_graph, file)

# Leiden Clustering

In [ ]:
labels, score, communities = run_clustering(
    kNN_adjacency_matrix,
    config,
)

# Select Communities

In [ ]:
communities_selected = select_communities(kNN_graph, communities, config)
top_m = len(communities_selected)
print(top_m)

# Create graph folder

In [ ]:
os.makedirs(config.graph_directory, exist_ok=True)

# Robustness Analysis

In [ ]:
big_objects()
ari_scores = run_bootstrap_robustness(kNN_adjacency_matrix, config)

In [ ]:
if "ari_scores" not in locals():
    ari_scores = load_bootstrap_scores(config)

median_ari = np.median(ari_scores)
mean_ari = np.mean(ari_scores)
std_ari = np.std(ari_scores)
print("Median ARI:", median_ari)
print("Mean ARI:", mean_ari)
print("STD of ARI:", std_ari)

In [ ]:
font_size = 20
tick_font_size = 16
plt.rcParams.update(
    {
        "font.size": font_size,
        "axes.titlesize": font_size,
        "axes.labelsize": font_size,
        "xtick.labelsize": tick_font_size,
        "ytick.labelsize": tick_font_size,
        "legend.fontsize": font_size,
        "figure.titlesize": font_size,
        "legend.loc": "best",
    }
)

plt.figure(figsize=(6, 4))
plt.hist(ari_scores, bins=15, edgecolor="black")
plt.axvline(
    median_ari,
    linestyle="--",
    linewidth=2,
    label=f"Median = {median_ari:.3f}",
)
plt.xlabel("Adjusted Rand Index (ARI)")
plt.ylabel("Sample Count")
plt.legend(fontsize=14)
plt.tight_layout()
plt.savefig(
    f"{config.graph_directory}/ari_stability_plot.png",
    dpi=300,
)
plt.show()